---
# <div align="center"><font color='blue'> COSC 2779/2972 | Deep Learning </font></div>
## <div align="center"><font color='blue'> Week 6 Lab: End-to-End Computer Vision Workflow</font></div>
---

## 🎯 Learning Objectives

By the end of this lab, you should be able to:

- Define realistic performance targets (metric and target value) using literature, domain knowledge, and EDA
- Build and justify a strong baseline model using established approaches
- Improve models by **reading the training signals**; not guessing
- Tune relevant hyperparameters **at each step**, not as a separate phase
- Perform error analysis, per-class analysis, and identify model biases
- Evaluate fairly on a held-out test set as a final certificate of generalisation

**Prerequisites:** Weeks 3–5 (optimisation, CNNs, transfer learning)

**Duration:** ~2–3 hours

---

## Lab Philosophy

This lab follows the practical ML workflow advocated by Andrew Ng in *Machine Learning Yearning* (free book: [deeplearning.ai/machine-learning-yearning-book](https://info.deeplearning.ai/machine-learning-yearning-book)).

The core principles:

> *"Don't start off trying to design and build the perfect system. Instead, build and train a basic system quickly. Then use error analysis to help you identify the most promising directions and iteratively improve your algorithm from there."*
> — Andrew Ng, *Machine Learning Yearning*, Ch. 13

**An important caveat:** "basic" does not mean "naive." Your first system should be the simplest approach that is **known to be competitive** for the problem type. A baseline so weak that it fails to learn anything useful gives you no signal to work with. For image classification in 2025, that means starting with a pretrained model and transfer learning — not a two-layer CNN from scratch.

> *"Most machine learning problems leave clues that tell you what's useful to try, and what's not useful to try. Learning to read those clues will save you months or years of development time."*
> — Andrew Ng, *Machine Learning Yearning*, Ch. 1

Every change you make must be justified by evidence — a training curve, an error pattern, a metric gap. Random experimentation is not engineering.

> 🔑 **The workflow:** Define targets → Explore data → Strong baseline → Read signals → Improve with evidence → Tune at each step → Analyse errors → Certify on test set

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, random_split
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### Training Utilities (from previous weeks)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        outputs = model(x)
        loss = criterion(outputs, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * y.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.inference_mode():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            loss = criterion(outputs, y)
            total_loss += loss.item() * y.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total


def run_experiment(model, optimizer, criterion, train_loader, val_loader,
                   epochs=10, name='Model'):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        if (epoch + 1) % max(1, epochs // 5) == 0 or epoch == 0:
            print(f'  [{name}] Epoch {epoch+1}/{epochs} — '
                  f'Train: {train_loss:.4f} / {train_acc:.4f} | '
                  f'Val: {val_loss:.4f} / {val_acc:.4f}')
    return history


def plot_experiments(results, show_train=True, show_val=True):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, hist in results.items():
        epochs = range(1, len(hist['train_loss']) + 1)
        if show_train:
            axes[0].plot(epochs, hist['train_loss'], label=f'{name} (Train)', marker='o', markersize=3)
            axes[1].plot(epochs, hist['train_acc'], label=f'{name} (Train)', marker='o', markersize=3)
        if show_val:
            axes[0].plot(epochs, hist['val_loss'], label=f'{name} (Val)', linestyle='--', marker='x', markersize=3)
            axes[1].plot(epochs, hist['val_acc'], label=f'{name} (Val)', linestyle='--', marker='x', markersize=3)
    for ax, ylabel in zip(axes, ['Loss', 'Accuracy']):
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel); ax.set_title(ylabel)
        ax.legend(fontsize=8); ax.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

---
## Dataset: Intel Image Classification

The **Intel Image Classification** dataset contains ~25,000 images of natural scenes (150×150 pixels) across 6 categories: **buildings, forest, glacier, mountain, sea, street**.

The dataset comes pre-split: 
- `seg_train` (~14k), 
- `seg_test` (~3k), 
- `seg_pred` (~7k unlabelled - we will not be using this data, they dont have labels so not useful as a test set).

You will need an account in opendatasets to download the data using the code below. if not, use the link in code to download directly and add to your google drive. 

In [ ]:
# Download from Kaggle using opendatasets
!pip install -q opendatasets

import opendatasets as od
od.download('https://www.kaggle.com/datasets/puneet6060/intel-image-classification')

In [ ]:
DATA_DIR = 'intel-image-classification'
TRAIN_DIR = os.path.join(DATA_DIR, 'seg_train', 'seg_train')
TEST_DIR = os.path.join(DATA_DIR, 'seg_test', 'seg_test')

# Verify structure
for split_name, split_dir in [('Training', TRAIN_DIR), ('Test', TEST_DIR)]:
    print(f'{split_name}:')
    for cls in sorted(os.listdir(split_dir)):
        cls_path = os.path.join(split_dir, cls)
        if os.path.isdir(cls_path):
            n = len([f for f in os.listdir(cls_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
            print(f'  {cls}: {n}')
    print()

### Creating a Validation Set

> 🚨 The test set (`seg_test`) must remain untouched during development. We split training data into train and validation sets.

In [ ]:
CLASS_NAMES = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
NUM_CLASSES = len(CLASS_NAMES)

base_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
])

full_train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=base_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=base_transform)

torch.manual_seed(42)
n_train = int(0.8 * len(full_train_dataset))
n_val = len(full_train_dataset) - n_train
train_subset, val_subset = random_split(full_train_dataset, [n_train, n_val])

print(f'Training:   {len(train_subset)}')
print(f'Validation: {len(val_subset)}')
print(f'Test:       {len(test_dataset)} (DO NOT TOUCH until final evaluation)')
print(f'Classes:    {full_train_dataset.classes}')

---
## Part 1: Define Performance Targets 

Before writing any model code, decide **what success looks like**.

> *"Having a single-number evaluation metric allows you to sort all your models according to their performance on this metric, and quickly decide what is working best."*
> — Andrew Ng, *ML Yearning*, Ch. 8

### Step 1: Choose a Target Metric

A single number that judges every experiment. Without it, you can't compare models or make decisions.

### Step 2: Set a Target Value

A target value defines **when development stops**. Set it using literature benchmarks, domain requirements, and baseline performance.

### Known Benchmarks for Intel Image Classification

| Approach | Accuracy |
|---|---|
| Random guessing (6 classes) | ~16.7% |
| Simple CNN (no tuning) | ~75–80% |
| CNN + augmentation + regularisation | ~85–88% |
| ResNet-50 transfer learning (fine-tune) | ~91% |
| ResNet-34 + 1-cycle + fine-tune | ~94% |

*Sources: Kaggle notebooks, GitHub repos, various blog posts*

**Key confusions reported in literature:**
- glacier ↔ mountain (both have snow)
- buildings ↔ street (both are urban)
- sea ↔ glacier (blue/white tones)
- forest is easiest to classify (distinctive green)

In [ ]:
# ============================================
# DEFINE YOUR TARGETS HERE
# ============================================

TARGET_METRIC = 'accuracy'  # TODO: Choose and justify
TARGET_VALUE = 0.88         # TODO: Set a realistic target value

print(f'Target metric: {TARGET_METRIC}')
print(f'Target value:  {TARGET_VALUE:.1%}')

Justification (fill in):
- Why this metric?

- Why this target value?


### 🤔 Questions

1. Is accuracy appropriate for this dataset, or would F1-score be better? Why?
2. What risks come from setting a target that is too high? Too low?
3. Why is it important to define the target **before** starting model development?

*Double-click here to write your answers for Part 0...*

---
## Part 2: Exploratory Data Analysis 


Before modelling, understand your data. EDA informs every subsequent decision.

In [ ]:
# Show sample images from each class
fig, axes = plt.subplots(3, 6, figsize=(16, 8))
fig.suptitle('Sample Images per Class', fontsize=14)

class_samples = {i: [] for i in range(NUM_CLASSES)}
for img, label in full_train_dataset:
    if len(class_samples[label]) < 3:
        class_samples[label].append(img)
    if all(len(v) >= 3 for v in class_samples.values()):
        break

for col in range(6):
    for row in range(3):
        axes[row, col].imshow(class_samples[col][row].permute(1, 2, 0).numpy())
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(CLASS_NAMES[col], fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution
train_labels = [full_train_dataset.targets[i] for i in train_subset.indices]
val_labels = [full_train_dataset.targets[i] for i in val_subset.indices]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, labels, title in zip(axes, [train_labels, val_labels], ['Training', 'Validation']):
    counts = Counter(labels)
    ax.bar([CLASS_NAMES[k] for k in sorted(counts)],
           [counts[k] for k in sorted(counts)],
           color='steelblue', edgecolor='white')
    ax.set_title(f'{title} Set'); ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Dataset statistics for normalisation
sample_images = torch.stack([full_train_dataset[i][0] for i in range(500)])
print(f'Image shape: {sample_images[0].shape}')
print(f'Mean per channel: {[f"{v:.4f}" for v in sample_images.mean(dim=(0,2,3)).tolist()]}')
print(f'Std per channel:  {[f"{v:.4f}" for v in sample_images.std(dim=(0,2,3)).tolist()]}')

### 🤔 Questions

1. Is the dataset balanced?
2. Which class pairs do you expect the model to confuse most? Why?
3. What preprocessing and augmentation would you apply?
4. Are there ambiguous images (e.g., an image with both buildings and a street)?

*Double-click here to write your answers for Part 1...*

---
## Part 3: Establish a Strong Baseline

### What Makes a Good Baseline?

> *"Don't start off trying to design and build the perfect system. Instead, build and train a basic system quickly—perhaps in just a few days. Even if the basic system is far from the 'best' system you can build, it is valuable to examine how the basic system functions: you will quickly find clues that show you the most promising directions in which to invest your time."*
> — Andrew Ng, *ML Yearning*, Ch. 13

A baseline is **not** a toy model. It should be something you know gives you the **best chance of solving the problem quickly**. For image classification in 2025, a pretrained model with transfer learning is the established strong starting point. Using a simple 3-layer CNN when pretrained models are available wastes time and tells you nothing useful.

The baseline should be:
- The simplest approach **that is known to be competitive** for this type of problem
- Quick to set up (no extensive tuning)
- Trained with reasonable defaults

For scene classification with ~11k training images, that means **ResNet-18 with transfer learning** — not a CNN from scratch.

### Setting Up Transforms

In [ ]:
# ImageNet normalisation (required for pretrained models)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform_v1 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds_v1 = datasets.ImageFolder(TRAIN_DIR, transform=train_transform_v1)
val_ds = datasets.ImageFolder(TRAIN_DIR, transform=val_transform)
train_ds_v1 = Subset(train_ds_v1, train_subset.indices)
val_ds = Subset(val_ds, val_subset.indices)

train_loader_v1 = DataLoader(train_ds_v1, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

### Baseline: ResNet-18 with Transfer Learning

We use the two-step transfer learning workflow from Week 5:
1. **Linear probing** — freeze backbone, train only the head
2. **Fine-tuning** — unfreeze backbone with a reduced learning rate

**Why this model.**  
Write justification:

In [ ]:
def create_resnet18(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def freeze_backbone(model):
    for name, param in model.named_parameters():
        if 'fc' not in name:
            param.requires_grad = False

def unfreeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = True

def count_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'  Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)')

In [ ]:
# Step 1: Linear probing
torch.manual_seed(0)
baseline = create_resnet18(NUM_CLASSES).to(device)
freeze_backbone(baseline)

print('Step 1: Linear Probing')
count_params(baseline)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, baseline.parameters()), lr=1e-3)
criterion = nn.CrossEntropyLoss()

all_results = {}
all_results['v1a: Linear Probing'] = run_experiment(
    baseline, optimizer, criterion, train_loader_v1, val_loader,
    epochs=5, name='v1a: Probing'
)

In [ ]:
# Step 2: Fine-tuning (continue from the same model)
unfreeze_backbone(baseline)

print('\nStep 2: Fine-Tuning')
count_params(baseline)

optimizer = optim.Adam(baseline.parameters(), lr=1e-5)

all_results['v1b: Fine-Tuning'] = run_experiment(
    baseline, optimizer, criterion, train_loader_v1, val_loader,
    epochs=10, name='v1b: Fine-Tune'
)

plot_experiments(all_results)

best_val = max(all_results['v1b: Fine-Tuning']['val_acc'])
print(f'\nBaseline val accuracy: {best_val:.4f} | Target: {TARGET_VALUE:.4f} | '
      f'{"✅ TARGET MET" if best_val >= TARGET_VALUE else "❌ Below target — analyse and improve"}')

### 🤔 Questions

1. What is the baseline validation accuracy? How far is it from your target?
2. Look at the training curves: is the model overfitting, underfitting, or is there an optimisation issue?
3. Based on the curves, what is the **first** thing you should try to improve?
4. Why did we use a pretrained model as the baseline rather than a simple CNN from scratch?

*Double-click here to write your answers for Part 2...*

---
## Part 3: Reading Training Signals & Evidence-Based Improvement

**You may not have time to finish all parts of this section in the lab. Just do what you can in the lab, and remainder can be done later by your own**

The training curves tell you **what to do next**. This is the core skill of practical ML.

> *"Your team has a lot of ideas... If you choose well among these possible directions, you'll lead your company to success. If you choose poorly, you might waste months."*
> — Andrew Ng, *ML Yearning*, Ch. 1

### The Decision Framework

| What you observe | Diagnosis | What to try |
|---|---|---|
| Train loss ↓, Val loss ↓ | Good learning | Continue, or raise target |
| Train loss ↓, Val loss ↑ | **Overfitting** | Augmentation, dropout, weight decay |
| Both losses stay high | **Underfitting** | Increase capacity, increase LR, train longer |
| Loss oscillates wildly | **Optimisation issue** | Reduce LR, try different optimiser |
| Train ≈ Val, both moderate | **Capacity limited** | Larger model, more data |

### 🚨 Two Rules

1. **Every change must be justified by evidence** from the training curves.
2. **Tune the relevant hyperparameters at each step** — don't defer tuning to a separate phase. If you add dropout, tune the dropout rate *now*. If you add augmentation, experiment with the augmentation strength *now*.

### Improvement Attempt 1: Data Augmentation

👉 **Note:** When you add augmentation, tune its parameters at this step. Try different augmentation strengths and choose the best.

In [ ]:
# ============================================
# ITERATION LOG — Attempt 1
# ============================================
# Signal:     ...
# Diagnosis:  ...
# Action:     Adding data augmentation + tuning augmentation strength
# ============================================

# Try two augmentation levels
aug_configs = {
    'Light Aug': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    'Strong Aug': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
}

aug_results = {}
for aug_name, aug_transform in aug_configs.items():
    aug_ds = datasets.ImageFolder(TRAIN_DIR, transform=aug_transform)
    aug_ds = Subset(aug_ds, train_subset.indices)
    aug_loader = DataLoader(aug_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)

    torch.manual_seed(0)
    model = create_resnet18(NUM_CLASSES).to(device)
    freeze_backbone(model)
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    run_experiment(model, opt, criterion, aug_loader, val_loader, epochs=3, name=f'{aug_name} probe')

    unfreeze_backbone(model)
    opt = optim.Adam(model.parameters(), lr=1e-5)
    aug_results[aug_name] = run_experiment(
        model, opt, criterion, aug_loader, val_loader,
        epochs=10, name=f'{aug_name} fine-tune'
    )
    print(f'  {aug_name} best val acc: {max(aug_results[aug_name]["val_acc"]):.4f}\n')

plot_experiments(aug_results)

In [ ]:
# Select the best augmentation and record it
best_aug_name = max(aug_results, key=lambda k: max(aug_results[k]['val_acc']))
print(f'Best augmentation: {best_aug_name}')
best_aug_transform = aug_configs[best_aug_name]

# Store in all_results
all_results['v2: +Augmentation'] = aug_results[best_aug_name]

best_val = max(all_results['v2: +Augmentation']['val_acc'])
print(f'Best val accuracy: {best_val:.4f} | Target: {TARGET_VALUE:.4f} | '
      f'{"✅ TARGET MET" if best_val >= TARGET_VALUE else "❌ Below target"}')

In [ ]:
# Result:    ...
# Next step: ...

### Improvement Attempt 2: Regularisation + Tuning

If still overfitting, add regularisation. **Tune the strength at this step.**

In [ ]:
# ============================================
# ITERATION LOG — Attempt 2
# ============================================
# Signal:     ...
# Diagnosis:  ...
# Action:     Adding weight decay + tuning its value
# ============================================

aug_ds = datasets.ImageFolder(TRAIN_DIR, transform=best_aug_transform)
aug_ds = Subset(aug_ds, train_subset.indices)
aug_loader = DataLoader(aug_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)

wd_results = {}
for wd in [0.0, 1e-4, 1e-3]:
    torch.manual_seed(0)
    model = create_resnet18(NUM_CLASSES).to(device)
    freeze_backbone(model)
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    run_experiment(model, opt, criterion, aug_loader, val_loader, epochs=3, name=f'wd={wd} probe')

    unfreeze_backbone(model)
    opt = optim.Adam(model.parameters(), lr=1e-5, weight_decay=wd)
    wd_results[f'wd={wd}'] = run_experiment(
        model, opt, criterion, aug_loader, val_loader,
        epochs=10, name=f'wd={wd}'
    )
    print(f'  wd={wd} best val acc: {max(wd_results[f"wd={wd}"]["val_acc"]):.4f}\n')

plot_experiments(wd_results, show_train=False)

# Select best
best_wd_name = max(wd_results, key=lambda k: max(wd_results[k]['val_acc']))
print(f'Best weight decay: {best_wd_name}')
all_results['v3: +Aug+WD'] = wd_results[best_wd_name]

best_val = max(all_results['v3: +Aug+WD']['val_acc'])
print(f'Best val accuracy: {best_val:.4f} | Target: {TARGET_VALUE:.4f} | '
      f'{"✅ TARGET MET" if best_val >= TARGET_VALUE else "❌ Below target"}')

In [ ]:
# Result:    ...
# Next step: ...

### Improvement Attempt 3: Learning Rate Tuning

The fine-tuning learning rate is one of the most impactful hyperparameters. Tune it now with your best setup so far.

In [ ]:
# ============================================
# ITERATION LOG — Attempt 3
# ============================================
# Signal:     ...
# Diagnosis:  ...
# Action:     Tuning the fine-tuning learning rate
# ============================================

best_wd_val = float(best_wd_name.split('=')[1])

lr_results = {}
for lr in [5e-6, 1e-5, 5e-5]:
    torch.manual_seed(0)
    model = create_resnet18(NUM_CLASSES).to(device)
    freeze_backbone(model)
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    run_experiment(model, opt, criterion, aug_loader, val_loader, epochs=3, name=f'lr={lr} probe')

    unfreeze_backbone(model)
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=best_wd_val)
    lr_results[f'lr={lr}'] = run_experiment(
        model, opt, criterion, aug_loader, val_loader,
        epochs=10, name=f'lr={lr}'
    )
    print(f'  lr={lr} best val acc: {max(lr_results[f"lr={lr}"]["val_acc"]):.4f}\n')

plot_experiments(lr_results, show_train=False)

# Select best
best_lr_name = max(lr_results, key=lambda k: max(lr_results[k]['val_acc']))
print(f'Best LR: {best_lr_name}')
all_results['v4: +Aug+WD+LR'] = lr_results[best_lr_name]

best_val = max(all_results['v4: +Aug+WD+LR']['val_acc'])
print(f'Best val accuracy: {best_val:.4f} | Target: {TARGET_VALUE:.4f} | '
      f'{"✅ TARGET MET" if best_val >= TARGET_VALUE else "❌ Below target"}')

In [ ]:
# Result:    ...
# Next step: ...

### 🤔 Questions (Part 3)

1. For each improvement, what signal prompted the change and did the result match your expectation?
2. Why is it important to tune hyperparameters *at each step* rather than deferring to a separate phase?
3. Which change had the biggest impact?

*Double-click here to write your answers for Part 3...*

---
## Part 4: Model Analysis

Once you've reached (or approached) your target, stop improving and start **understanding**.

> *"Error analysis refers to the process of examining dev set examples that your algorithm misclassified, so that you can understand the underlying causes of the errors. This can help you prioritize projects and inspire new directions."*
> — Andrew Ng, *ML Yearning*, Ch. 14

### Retrain Best Configuration

In [ ]:
# Retrain with the best settings found in Part 3
best_lr_val = float(best_lr_name.split('=')[1])

torch.manual_seed(0)
best_model = create_resnet18(NUM_CLASSES).to(device)
freeze_backbone(best_model)
opt = optim.Adam(filter(lambda p: p.requires_grad, best_model.parameters()), lr=1e-3)
run_experiment(best_model, opt, criterion, aug_loader, val_loader, epochs=3, name='Best probe')

unfreeze_backbone(best_model)
opt = optim.Adam(best_model.parameters(), lr=best_lr_val, weight_decay=best_wd_val)
run_experiment(best_model, opt, criterion, aug_loader, val_loader, epochs=10, name='Best fine-tune')

print('Best model trained.')

### Task 4.1 — Per-Class Performance

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

best_model.eval()
all_preds, all_labels = [], []
with torch.inference_mode():
    for x, y in val_loader:
        x = x.to(device)
        _, predicted = torch.max(best_model(x), 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

### Task 4.2 — Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
plt.title('Confusion Matrix — Validation Set')
plt.tight_layout()
plt.show()

### Task 4.3 — Error Analysis: Inspect Misclassified Images

In [ ]:
vis_dataset = datasets.ImageFolder(TRAIN_DIR, transform=transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
]))

errors = np.where(all_preds != all_labels)[0]
print(f'Misclassified: {len(errors)} / {len(all_labels)} ({len(errors)/len(all_labels)*100:.1f}%)')

n_show = min(16, len(errors))
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i, ax in enumerate(axes.flat):
    if i >= n_show:
        ax.axis('off'); continue
    global_idx = val_subset.indices[errors[i]]
    img, _ = vis_dataset[global_idx]
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(f'T:{CLASS_NAMES[all_labels[errors[i]]]}\nP:{CLASS_NAMES[all_preds[errors[i]]]}', fontsize=8)
    ax.axis('off')
plt.suptitle('Misclassified Images', fontsize=12)
plt.tight_layout()
plt.show()

### Task 4.4 — Hypothesis Testing

> *"Before investing a month on this task, I recommend that you first estimate how much it will actually improve the system's accuracy. Then you can more rationally decide if this is worth the development time."*
> — Andrew Ng, *ML Yearning*, Ch. 14

1. **Observe** the confusion matrix and misclassified images
2. **Hypothesise** a specific explanation
3. **Estimate** the potential ceiling (what % of errors does this category account for?)
4. **Test** with a targeted change
5. **Conclude**

In [ ]:
# ============================================
# HYPOTHESIS LOG
# ============================================
# Observation: ...
# Hypothesis:  ...
# Error ceiling: ...% of misclassifications are in this category
# Test:        ...
# Result:      ...
# Conclusion:  ...
# ============================================

### Task 4.5 — Bias and Fairness

Consider:
- **Per-class accuracy disparity:** Is 95% on forests but 75% on glaciers acceptable?
- **Ambiguous labels:** Some images contain both buildings and streets — the model is penalised for arguably correct answers.
- **Geographic bias:** Are scenes from all regions represented?
- **Deployment context:** Which class errors would be most harmful?

### 🤔 Questions (Part 4)

1. Which classes have the highest and lowest accuracy? Does this match your EDA predictions?
2. What are the most common confusion pairs? Are they reasonable?
3. What hypothesis did you form, and did testing it help?
4. If deployed in a real application, which errors would be most concerning?

*Double-click here to write your answers for Part 4...*

---
## Part 5: Final Evaluation — The Certificate

The **test set** has been untouched throughout. We now use it **exactly once**.

> 🚨 **This is the final certificate.** The number you get here is the one you report. No further development is allowed.

In [ ]:
test_ds = datasets.ImageFolder(TEST_DIR, transform=val_transform)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)

test_loss, test_acc = evaluate(best_model, test_loader, nn.CrossEntropyLoss())

print('=' * 60)
print('FINAL TEST SET EVALUATION')
print('=' * 60)
print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test Loss:     {test_loss:.4f}')
print(f'Target:        {TARGET_VALUE:.4f}')
print(f'Status:        {"✅ TARGET MET" if test_acc >= TARGET_VALUE else "❌ Below target"}')
print('=' * 60)

In [ ]:
best_model.eval()
test_preds, test_labels = [], []
with torch.inference_mode():
    for x, y in test_loader:
        x = x.to(device)
        _, predicted = torch.max(best_model(x), 1)
        test_preds.extend(predicted.cpu().numpy())
        test_labels.extend(y.numpy())

print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES))

### 🤔 Questions

1. How does test accuracy compare to validation accuracy? Is the gap acceptable?
2. If there is a gap, what might explain it?
3. Did you meet your target?

*Double-click here to write your answers for Part 5...*

---
## Development Summary

In [ ]:
print(f'{"Version":<25} {"Best Val Acc":<15} {"Final Train Acc":<15}')
print('=' * 55)
for name, hist in all_results.items():
    print(f'{name:<25} {max(hist["val_acc"]):<15.4f} {hist["train_acc"][-1]:<15.4f}')
print(f'\n{"FINAL TEST":<25} {test_acc:<15.4f}')
print(f'{"TARGET":<25} {TARGET_VALUE:<15.4f}')

---
## Part 6: Reflection

1. What was the most important decision you made? Why?
2. What mistake did you make, and what did you learn from it?
3. At which point did you get the biggest improvement?
4. How did the decision framework (observe → diagnose → act → tune) compare to how you would have approached this before the course?
5. Ng advises building the first system quickly and iterating. How did this principle work in practice during this lab?
6. What would you do differently if you started over?

*Double-click here to write your reflection...*

---
## Bonus (Optional)

- **Learning rate scheduling:** Try `CosineAnnealingLR` or `OneCycleLR`
- **Different backbone:** Try `resnet34`, `resnet50`, or `efficientnet_b0`
- **Ensemble:** Average predictions from 3 models with different seeds
- **Confidence analysis:** Inspect softmax probabilities on misclassified images

---
## References

- Andrew Ng, *Machine Learning Yearning* (2018). Free download: [deeplearning.ai/machine-learning-yearning-book](https://info.deeplearning.ai/machine-learning-yearning-book)
- Intel Image Classification dataset: [kaggle.com/datasets/puneet6060/intel-image-classification](https://www.kaggle.com/datasets/puneet6060/intel-image-classification)
- He et al., "Deep Residual Learning for Image Recognition" (2015). [arXiv:1512.03385](https://arxiv.org/abs/1512.03385)



---

## 🔑 Key Takeaway

> *"Most machine learning problems leave clues that tell you what's useful to try, and what's not useful to try. Learning to read those clues will save you months or years of development time."*
> — Andrew Ng, *ML Yearning*